# Multiple regression

A response can depend on more than one predictor. The model remains linear in the coefficients. The normal equations remain $X^{\mathsf T}X\,\beta = X^{\mathsf T}y$. What changes is the interpretation: each slope is the change in the mean response when that predictor increases by one unit and the other predictors are held fixed.


## The three-coefficient system

The table is

$$
\begin{array}{c|cccc}
x_1 & 1 & 0 & 1 & 2 \\
x_2 & 0 & 1 & 1 & 1 \\
\hline
y & 3 & 4 & 6 & 9
\end{array}
$$

The design matrix has columns $1$, $x_1$, and $x_2$. The Gram matrix is made of the column sums

$$
\begin{aligned}
n &= 4, & \sum x_1 &= 4, & \sum x_2 &= 3, \\
\sum x_1^2 &= 6, & \sum x_2^2 &= 3, & \sum x_1 x_2 &= 3,
\end{aligned}
$$

so

$$
X^{\mathsf T}X = \begin{bmatrix} 4 & 4 & 3 \\ 4 & 6 & 3 \\ 3 & 3 & 3 \end{bmatrix}.
$$

The right-hand side uses $y$:

$$
\begin{aligned}
\sum y &= 22, \\
\sum x_1 y &= 1\cdot 3 + 0\cdot 4 + 1\cdot 6 + 2\cdot 9 = 27, \\
\sum x_2 y &= 0\cdot 3 + 1\cdot 4 + 1\cdot 6 + 1\cdot 9 = 19.
\end{aligned}
$$

The system is

$$
\begin{aligned}
4b_0 + 4b_1 + 3b_2 &= 22, \\
4b_0 + 6b_1 + 3b_2 &= 27, \\
3b_0 + 3b_1 + 3b_2 &= 19.
\end{aligned}
$$

Subtract the first equation from the second: $2b_1 = 5$, so $b_1 = 5/2$. Divide the third by $3$:

$$
b_0 + b_1 + b_2 = \dfrac{19}{3}
\Longrightarrow b_0 + b_2 = \dfrac{19}{3} - \dfrac{5}{2} = \dfrac{23}{6}.
$$

Return to the first equation with $b_1 = 5/2$:

$$
4b_0 + 10 + 3b_2 = 22 \Longrightarrow 4b_0 + 3b_2 = 12.
$$

Substitute $b_0 = 23/6 - b_2$:

$$
4\left(\dfrac{23}{6} - b_2\right) + 3b_2 = 12
\Longrightarrow \dfrac{46}{3} - b_2 = 12
\Longrightarrow b_2 = \dfrac{10}{3}, \qquad b_0 = \dfrac{23}{6} - \dfrac{10}{3} = \dfrac{1}{2}.
$$

Thus

$$
\widehat y = \dfrac{1}{2} + \dfrac{5}{2}x_1 + \dfrac{10}{3}x_2.
$$

Holding $x_2$ fixed, one more unit of $x_1$ adds $5/2$ to the prediction. Holding $x_1$ fixed, one more unit of $x_2$ adds $10/3$.


## Residuals of that fit

$$
\begin{aligned}
\widehat y(1, 0) &= \dfrac{1}{2} + \dfrac{5}{2} = 3, \\
\widehat y(0, 1) &= \dfrac{1}{2} + \dfrac{10}{3} = \dfrac{23}{6}, \\
\widehat y(1, 1) &= 3 + \dfrac{10}{3} = \dfrac{19}{3}, \\
\widehat y(2, 1) &= \dfrac{1}{2} + 5 + \dfrac{10}{3} = \dfrac{53}{6}.
\end{aligned}
$$

$$
e = 0,\ \dfrac{1}{6},\ -\dfrac{1}{3},\ \dfrac{1}{6}, \qquad
\sum e_i = 0, \qquad
\mathrm{RSS} = \dfrac{1}{36} + \dfrac{4}{36} + \dfrac{1}{36} = \dfrac{1}{6}.
$$

The residuals are also orthogonal to both predictors, because every column of $X$ was given a normal equation. That check is left to the code.


In [1]:
# Solve the 3 by 3 normal equations and check the hand residuals.
from fractions import Fraction
import numpy as np
X = np.array([
    [1, 1, 0],
    [1, 0, 1],
    [1, 1, 1],
    [1, 2, 1],
], dtype=float)
y = np.array([3, 4, 6, 9], dtype=float)
beta = np.linalg.solve(X.T @ X, X.T @ y)
hand = np.array([1 / 2, 5 / 2, 10 / 3])
fitted = X @ hand
errors = y - fitted
print("python beta", beta)
print("fitted", fitted)
print("residuals", errors)
print("RSS", np.sum(errors ** 2))
print("X' e", X.T @ errors)
assert np.allclose(beta, hand)
assert np.allclose(fitted, [3, 23 / 6, 19 / 3, 53 / 6])
assert np.allclose(errors, [0, 1 / 6, -1 / 3, 1 / 6])
assert abs(np.sum(errors ** 2) - 1 / 6) < 1e-12
assert np.allclose(X.T @ errors, 0)


python beta [0.5        2.5        3.33333333]
fitted [3.         3.83333333 6.33333333 8.83333333]
residuals [ 0.          0.16666667 -0.33333333  0.16666667]
RSS 0.1666666666666668
X' e [-1.33226763e-15 -1.77635684e-15 -1.33226763e-15]


## Leaving a predictor out

A second table is built from the exact law $y = 1 + 2x_1 + x_2$:

$$
\begin{array}{c|cccc}
x_1 & 1 & 2 & 3 & 4 \\
x_2 & 1 & 3 & 2 & 4 \\
\hline
y & 4 & 8 & 9 & 13
\end{array}
$$

The full design has a perfect fit at $\beta = (1, 2, 1)^{\mathsf T}$, so its residual sum of squares is zero. The short regression of $y$ on $x_1$ alone uses

$$
\begin{aligned}
\sum x_1 &= 10, & \sum y &= 34, & \sum x_1 y &= 99, & \sum x_1^2 &= 30.
\end{aligned}
$$

$$
\begin{aligned}
b_1 &= \dfrac{4\cdot 99 - 10\cdot 34}{4\cdot 30 - 10^2} = \dfrac{56}{20} = \dfrac{14}{5}, \\
b_0 &= \dfrac{34 - \tfrac{14}{5}\cdot 10}{4} = \dfrac{6}{4} = \dfrac{3}{2}.
\end{aligned}
$$

The slope of $x_1$ moved from $2$ to $14/5$. The columns are not orthogonal after centering: $x_2$ and $x_1$ share variation, and the omitted coefficient $1$ is not zero, so the short slope absorbs part of the omitted predictor. This is omitted-variable bias, visible here as arithmetic rather than as a warning label.


In [2]:
# The full law is recovered. Dropping x2 changes the slope of x1.
from fractions import Fraction
import numpy as np
x1 = np.array([1.0, 2, 3, 4])
x2 = np.array([1.0, 3, 2, 4])
y = 1 + 2 * x1 + x2
full = np.column_stack([np.ones(4), x1, x2])
beta_full = np.linalg.solve(full.T @ full, full.T @ y)
n = 4
slope = Fraction(n * int(x1 @ y) - int(x1.sum()) * int(y.sum()), n * int(x1 @ x1) - int(x1.sum()) ** 2)
intercept = Fraction(int(y.sum()) - slope * int(x1.sum()), n)
print("full beta", beta_full)
print("short line", intercept, slope)
print("full RSS", np.sum((y - full @ beta_full) ** 2))
assert np.allclose(beta_full, [1, 2, 1])
assert intercept == Fraction(3, 2) and slope == Fraction(14, 5)
assert np.sum((y - full @ beta_full) ** 2) < 1e-12


full beta [1. 2. 1.]
short line 3/2 14/5
full RSS 7.888609052210118e-31


## Pitfall

The phrase "holding $x_2$ fixed" is a statement about the fitted equation. It is not a statement that the data set contains a row in which $x_1$ changed and $x_2$ did not, and it is not a license to read $5/2$ as a causal effect. If $x_1$ and $x_2$ always move together, the separate coefficients are still computable whenever $X$ has full rank, but the data never showed the experiment the sentence describes.


In [3]:
# The coefficient is a partial slope in the equation, printed beside the rows that exist.
import numpy as np
rows = np.array([[1, 0], [0, 1], [1, 1], [2, 1]], dtype=float)
print("observed pairs of (x1, x2)")
print(rows)
print("partial slope of x1", 5 / 2)
assert rows.shape == (4, 2)


observed pairs of (x1, x2)
[[1. 0.]
 [0. 1.]
 [1. 1.]
 [2. 1.]]
partial slope of x1 2.5


## Summary

- In $\widehat y = \tfrac{1}{2} + \tfrac{5}{2}x_1 + \tfrac{10}{3}x_2$, each slope holds the other predictor fixed inside the equation.
- The residuals of that fit are $0, 1/6, -1/3, 1/6$, and $\mathrm{RSS} = 1/6$.
- Dropping a predictor that is correlated with the kept predictor changes the kept slope. Here $2$ became $14/5$.
